In [1]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score
)

In [2]:
# loading trained MLP
class SimpleMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )

    def forward(self, x):

        return self.network(x)


In [3]:
model = SimpleMLP()

state_dict = torch.load(
    "mlp_model.pth",
    map_location=torch.device("cpu"),
    weights_only=True
)

model.load_state_dict(state_dict)

model.eval()

print("MLP Loaded")

MLP Loaded


In [4]:
# loading scaler
scaler = joblib.load(
    "mlp_scaler.pkl"
)

print("Scaler Loaded")

Scaler Loaded


In [5]:
# creating dataset to feed to MLP
df = pd.read_csv("unknown_svm_dataset.csv")

mlp_df = df[
    [
        "quality_score",
        "best_similarity",
        "margin",
        "label"
    ]
]

mlp_df.to_csv(
    "unknown_mlp.csv",
    index=False
)

print(mlp_df.shape)

mlp_df.head()

(104, 4)


,quality_score,best_similarity,margin,label
0,35.787743,0.218973,0.026110,0
1,19.389027,0.213200,0.022957,0
2,32.406471,0.216415,0.017757,0
3,15.262703,0.230025,0.057809,0
4,14.295137,0.180464,0.031413,0


In [6]:
# running MLP
df = pd.read_csv(
    "unknown_mlp.csv"
)

X = df[
    [
        "quality_score",
        "best_similarity",
        "margin"
    ]
]

y_true = df["label"]

X_scaled = scaler.transform(X)

X_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32
)

with torch.no_grad():

    outputs = model(X_tensor)

    probabilities = torch.softmax(
        outputs,
        dim=1
    )

    y_pred = torch.argmax(
        outputs,
        dim=1
    ).numpy()

df["mlp_decision"] = y_pred

df.to_csv(
    "unknown_mlp_predictions.csv",
    index=False
)

print("Saved unknown_mlp_predictions.csv")

Saved unknown_mlp_predictions.csv


In [7]:
# metrics
correct_rejects = (
    y_pred == 0
).sum()

false_accepts = (
    y_pred == 1
).sum()

TRR = (
    correct_rejects
    /
    len(y_pred)
)

FAR = (
    false_accepts
    /
    len(y_pred)
)

print("\n===== UNKNOWN TEST RESULTS =====\n")

print("Total:", len(y_pred))

print("Correct Rejects:", correct_rejects)

print("False Accepts:", false_accepts)

print("TRR:", round(TRR, 4))

print("FAR:", round(FAR, 4))


===== UNKNOWN TEST RESULTS =====

Total: 104
Correct Rejects: 104
False Accepts: 0
TRR: 1.0
FAR: 0.0


In [8]:
# confusion matrix
cm_mlp = confusion_matrix(
    y_true,
    y_pred,
    labels=[0,1]
)

print("\nConfusion Matrix")

print(cm_mlp)


Confusion Matrix
[[104   0]
 [  0   0]]


In [9]:
# fixed threshold comparison
threshold = 0.6
threshold_pred = (
    df["best_similarity"] >= threshold
).astype(int)

In [10]:
threshold_correct_rejects = (
    threshold_pred == 0
).sum()

threshold_false_accepts = (
    threshold_pred == 1
).sum()

threshold_TRR = (
    threshold_correct_rejects
    /
    len(threshold_pred)
)

threshold_FAR = (
    threshold_false_accepts
    /
    len(threshold_pred)
)

print("\n===== THRESHOLD RESULTS =====\n")

print("Threshold:", threshold)

print("Correct Rejects:", threshold_correct_rejects)

print("False Accepts:", threshold_false_accepts)

print("TRR:", round(threshold_TRR, 4))

print("FAR:", round(threshold_FAR, 4))


===== THRESHOLD RESULTS =====

Threshold: 0.6
Correct Rejects: 104
False Accepts: 0
TRR: 1.0
FAR: 0.0


In [11]:
# comparison tables
svm_TRR = 1.0
svm_FAR = 0.0

comparison = pd.DataFrame(
    {
        "Method": [
            "Threshold",
            "SVM",
            "MLP"
        ],

        "TRR": [
            threshold_TRR,
            svm_TRR,
            TRR
        ],

        "FAR": [
            threshold_FAR,
            svm_FAR,
            FAR
        ],

        "Correct Rejects": [
            threshold_correct_rejects,
            104,
            correct_rejects
        ],

        "False Accepts": [
            threshold_false_accepts,
            0,
            false_accepts
        ]
    }
)

print("\n===== FINAL COMPARISON =====\n")

comparison


===== FINAL COMPARISON =====



,Method,TRR,FAR,Correct Rejects,False Accepts
0,Threshold,1.0,0.0,104,0
1,SVM,1.0,0.0,104,0
2,MLP,1.0,0.0,104,0
